# DINO Attention

Compare attention heads and layers. Is a thresholded attention map a ground-truth mask?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://github.com/facebookresearch/dino

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "timm==1.0.22"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
# Load the ViT-Small model trained with DINO
model_name = 'vit_small_patch16_224.dino'
model = timm.create_model(model_name, pretrained=True, num_classes=0)  # num_classes=0 → feature extractor
model = model.to(device)
model.eval()

print(f"✅ Model loaded: {model_name}")
print(f"   - Patch size: 16x16")
print(f"   - Input size: 224x224")
print(f"   - Embedding dim: {model.embed_dim}")
print(f"   - Number of heads: {model.blocks[0].attn.num_heads}")
print(f"   - Number of layers: {len(model.blocks)}")

# Check the number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"   - Total parameters: {total_params / 1e6:.1f}M")

In [ ]:
# Use the model-specific transform provided by timm
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

print("📐 Transform pipeline:")
print(transform)

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
def model_view(image):
    """Display exactly the resized, center-cropped input seen by DINO."""
    tensor = transform(image).cpu()
    mean = torch.tensor(data_config["mean"])[:, None, None]
    std = torch.tensor(data_config["std"])[:, None, None]
    pixels = ((tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(pixels)

def get_attention_maps(model, img_tensor, device):
    """
    Extract attention maps from every layer of the ViT.
    
    Returns:
        attentions: list of (1, num_heads, num_tokens, num_tokens) tensors
    """
    model.eval()
    attentions = []
    hooks = []
    
    def hook_fn(module, input, output):
        # Capture the attention weight from timm's Attention module
        # Run the forward directly to obtain the attention weight
        B, N, C = input[0].shape
        qkv = module.qkv(input[0]).reshape(B, N, 3, module.num_heads, C // module.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        
        scale = (C // module.num_heads) ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.softmax(dim=-1)
        attentions.append(attn.detach().cpu())
    
    # Register a hook on every Attention module
    for block in model.blocks:
        hook = block.attn.register_forward_hook(hook_fn)
        hooks.append(hook)
    
    # Forward pass
    with torch.no_grad():
        _ = model(img_tensor.unsqueeze(0).to(device))
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    return attentions


def visualize_attention(img_path, model, transform, device, layer=-1):
    """
    Visualize the ViT attention map for an image.
    
    Args:
        layer: layer index to visualize (-1 = last layer)
    """
    # Load image
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img)
    
    # Extract attention
    attentions = get_attention_maps(model, img_tensor, device)
    
    # Attention from the last layer (CLS token → patches)
    attn = attentions[layer]  # (1, num_heads, N, N)
    num_heads = attn.shape[1]
    
    # Attention from CLS token (index 0) to patch tokens
    cls_attn = attn[0, :, 0, 1:]  # (num_heads, num_patches)
    
    # Reshape to spatial map (14x14 for 224/16)
    patch_size = 16
    num_patches_side = 224 // patch_size  # 14
    cls_attn = cls_attn.reshape(num_heads, num_patches_side, num_patches_side)
    
    return img, cls_attn, num_heads

print("✅ Attention extraction function defined")

## Demo code

Compare attention heads and layers. Is a thresholded attention map a ground-truth mask?

In [ ]:
image_path = f"{IMG_DIR}/dog/dog_000.jpg"
image, attention, heads = visualize_attention(image_path, model, transform, device)
mean_attention = attention.mean(0).numpy()
mask = mean_attention > np.percentile(mean_attention, 60)
plt.imshow(mask, cmap="gray")
plt.axis("off"); plt.show()

In [ ]:
# Visualize the attention map for a few sample images
sample_paths = []
for label in ['dog', 'cat']:
    files = sorted(os.listdir(f'{IMG_DIR}/{label}'))[:3]
    sample_paths.extend([f'{IMG_DIR}/{label}/{f}' for f in files])

for img_path in sample_paths[:4]:
    img, cls_attn, num_heads = visualize_attention(img_path, model, transform, device)
    
    # Mean over heads + per-head
    n_show_heads = min(6, num_heads)
    fig, axes = plt.subplots(2, n_show_heads + 1, figsize=(3 * (n_show_heads + 1), 6))
    
    label_name = 'Dog 🐶' if 'dog' in img_path else 'Cat 🐱'
    fig.suptitle(f'Attention Maps — {label_name} ({os.path.basename(img_path)})', 
                fontsize=14, fontweight='bold')
    
    # Row 1: Original + per-head attention (raw)
    img_resized = model_view(img)
    axes[0, 0].imshow(img_resized)
    axes[0, 0].set_title('Original', fontsize=10, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[1, 0].imshow(img_resized)
    # Mean attention overlay
    mean_attn = cls_attn.mean(0).numpy()
    mean_attn = (mean_attn - mean_attn.min()) / (mean_attn.max() - mean_attn.min() + 1e-8)
    mean_attn_resized = np.array(Image.fromarray((mean_attn * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
    axes[1, 0].imshow(mean_attn_resized, cmap='hot', alpha=0.6)
    axes[1, 0].set_title('Mean Attn\n(overlay)', fontsize=10, fontweight='bold')
    axes[1, 0].axis('off')
    
    for h in range(n_show_heads):
        attn_h = cls_attn[h].numpy()
        attn_h = (attn_h - attn_h.min()) / (attn_h.max() - attn_h.min() + 1e-8)
        attn_h_resized = np.array(Image.fromarray((attn_h * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
        
        # Raw attention
        axes[0, h+1].imshow(attn_h_resized, cmap='inferno')
        axes[0, h+1].set_title(f'Head {h}', fontsize=10)
        axes[0, h+1].axis('off')
        
        # Overlay on image
        axes[1, h+1].imshow(img_resized)
        axes[1, h+1].imshow(attn_h_resized, cmap='hot', alpha=0.6)
        axes[1, h+1].set_title(f'Head {h}\n(overlay)', fontsize=10)
        axes[1, h+1].axis('off')
    
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# Compare mean attention across all layers for one image
sample_img_path = sample_paths[0]
img = Image.open(sample_img_path).convert('RGB')
img_tensor = transform(img)

attentions = get_attention_maps(model, img_tensor, device)
num_layers = len(attentions)

fig, axes = plt.subplots(2, (num_layers + 1) // 2 + 1, figsize=(24, 8))
fig.suptitle(f'Attention Across All {num_layers} Layers (Mean over Heads)', 
            fontsize=16, fontweight='bold')

img_resized = model_view(img)

# Original image in the first cell
axes[0, 0].imshow(img_resized)
axes[0, 0].set_title('Original', fontsize=10, fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for layer_idx in range(num_layers):
    row = layer_idx // ((num_layers + 1) // 2)
    col = layer_idx % ((num_layers + 1) // 2) + 1
    
    attn = attentions[layer_idx]
    cls_attn = attn[0, :, 0, 1:].mean(0)  # Mean over heads
    cls_attn = cls_attn.reshape(14, 14).numpy()
    cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
    
    attn_resized = np.array(Image.fromarray((cls_attn * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
    
    axes[row, col].imshow(img_resized)
    axes[row, col].imshow(attn_resized, cmap='hot', alpha=0.6)
    axes[row, col].set_title(f'Layer {layer_idx + 1}', fontsize=10)
    axes[row, col].axis('off')

# Hide the remaining empty cells
for ax in axes.flat:
    if not ax.get_title() and not ax.images:
        ax.axis('off')

plt.tight_layout()
plt.show()

print("\n💡 Observation: shallow layers attend to broad regions,")
print("   while deeper layers focus their attention on the object (the animal)!")

In [ ]:
# Perform simple segmentation using DINO's last-layer attention
fig, axes = plt.subplots(len(sample_paths[:4]), 4, figsize=(16, 4 * len(sample_paths[:4])))
fig.suptitle('🎯 DINO Attention-based Foreground Segmentation', fontsize=16, fontweight='bold')

for row, img_path in enumerate(sample_paths[:4]):
    img = model_view(Image.open(img_path).convert('RGB'))
    img_tensor = transform(Image.open(img_path).convert('RGB'))
    
    attentions = get_attention_maps(model, img_tensor, device)
    
    # Mean over all heads of the last layer
    cls_attn = attentions[-1][0, :, 0, 1:].mean(0).reshape(14, 14).numpy()
    
    # Normalize
    cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
    
    # Threshold for segmentation
    threshold = np.percentile(cls_attn, 60)
    mask = cls_attn > threshold
    mask_resized = np.array(Image.fromarray(mask.astype(np.uint8) * 255).resize((224, 224), Image.NEAREST)) / 255.0
    attn_resized = np.array(Image.fromarray((cls_attn * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
    
    # Masked image
    img_arr = np.array(img) / 255.0
    masked_img = img_arr * mask_resized[:, :, np.newaxis]
    
    # Plot
    axes[row, 0].imshow(img)
    axes[row, 0].set_title('Original', fontsize=11)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(attn_resized, cmap='inferno')
    axes[row, 1].set_title('Attention Map', fontsize=11)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(mask_resized, cmap='gray')
    axes[row, 2].set_title('Binary Mask', fontsize=11)
    axes[row, 2].axis('off')
    
    axes[row, 3].imshow(masked_img)
    axes[row, 3].set_title('Segmented', fontsize=11)
    axes[row, 3].axis('off')

plt.tight_layout()
plt.show()

print("\n💡 DINO's attention naturally captures object regions without any explicit supervision.")
print("   This is one of the key findings of the DINO paper!")

## Interpretation

Attention thresholds produce a heuristic foreground mask.

Compare the output with the evidence in the input. A generated answer can be wrong.